# Credit Card Fraud Detection with Gaussian Naive Bayes

This project evaluates a Gaussian Naive Bayes classifier for detecting
fraudulent transactions in an imbalanced dataset.

The workflow includes:
- stratified train/validation/test splitting
- feature scaling
- Gaussian Naive Bayes classification
- evaluation with precision, recall, F1-score and ROC-AUC
- decision-threshold selection using the validation set
- final evaluation on an independent test set

## 1. Imports

In [107]:
import pandas as pd

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_auc_score,
    average_precision_score
)

'data/card_transdata.csv'

## 2. Data Loading and Exploration

In [108]:
def load_data(features):
    df = pd.read_csv("data/card_transdata.csv")

    print(df.shape)
    print(df.head())
    df.info()
    print(df.isnull().sum())

    print("\nDistribución:")
    print(df["fraud"].value_counts())

    print("\nProporción:")
    print(df["fraud"].value_counts(normalize=True))

    X = df[features].values
    y = df["fraud"]

    return X, y

## 3. Model Training and Evaluation



In [109]:
def train_and_evaluate(X, y):
    ######### Separar, escalar y entrenar
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.2,
        random_state=42,
        stratify=y_train_val
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    gnb = GaussianNB()
    gnb.fit(X_train_scaled, y_train)

    ######### Probabilidades en VALIDATION
    y_prob_val = gnb.predict_proba(X_val_scaled)[:, 1]

    ######### Pruebas de Threshold
    thresholds = [
        0.15,
        0.175,
        0.20,
        0.225,
        0.25,
        0.5
    ]

    for threshold in thresholds:

        y_pred_val = (y_prob_val >= threshold).astype(int)

        accuracy_val = accuracy_score(y_val, y_pred_val)
        precision_val = precision_score(y_val, y_pred_val)
        recall_val = recall_score(y_val, y_pred_val)
        f1_val = f1_score(y_val, y_pred_val)

        print(f"\nThreshold: {threshold}")
        print(f"Accuracy: {accuracy_val:.4f}")
        print(f"Precision: {precision_val:.4f}")
        print(f"Recall: {recall_val:.4f}")
        print(f"F1 Score: {f1_val:.4f}")


    ######### Métricas generales de VALIDATION
    roc_auc_val = roc_auc_score(y_val, y_prob_val)
    average_precision_val = average_precision_score(y_val, y_prob_val)

    print(f"\nValidation ROC-AUC: {roc_auc_val}")
    print(f"\nValidation Average Precision: {average_precision_val:.4f}")


    ######### Threshold seleccionado
    best_threshold = 0.225

    y_pred_val = (y_prob_val >= best_threshold).astype(int)

    print(f"\nSelected Threshold: {best_threshold}")

    print("\nValidation Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred_val))

    print("\nValidation Classification Report:")
    print(classification_report(y_val, y_pred_val))


    ######### Evaluación FINAL en TEST
    X_test_scaled = scaler.transform(X_test)

    y_prob_test = gnb.predict_proba(X_test_scaled)[:, 1]

    y_pred_test = (
        y_prob_test >= best_threshold
    ).astype(int)

    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test)
    recall_test = recall_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test)

    roc_auc_test = roc_auc_score(y_test, y_prob_test)
    average_precision_test = average_precision_score(y_test, y_prob_test)

    print("\nFINAL TEST RESULTS")
    print(f"Threshold: {best_threshold}")
    print(f"Accuracy: {accuracy_test:.4f}")
    print(f"Precision: {precision_test:.4f}")
    print(f"Recall: {recall_test:.4f}")
    print(f"F1 Score: {f1_test:.4f}")
    print(f"ROC-AUC: {roc_auc_test:.4f}")
    print(f"Average Precision: {average_precision_test:.4f}")

    print("\nTest Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_test))

    print("\nTest Classification Report:")
    print(classification_report(y_test, y_pred_test))

## 4. Run Experiment


In [110]:
features = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

X, y = load_data(features)

train_and_evaluate(X, y)

(1000000, 8)
   distance_from_home  distance_from_last_transaction  \
0           57.877857                        0.311140   
1           10.829943                        0.175592   
2            5.091079                        0.805153   
3            2.247564                        5.600044   
4           44.190936                        0.566486   

   ratio_to_median_purchase_price  repeat_retailer  used_chip  \
0                        1.945940              1.0        1.0   
1                        1.294219              1.0        0.0   
2                        0.427715              1.0        0.0   
3                        0.362663              1.0        1.0   
4                        2.222767              1.0        1.0   

   used_pin_number  online_order  fraud  
0              0.0           0.0    0.0  
1              0.0           0.0    0.0  
2              0.0           1.0    0.0  
3              0.0           1.0    0.0  
4              0.0           1.0    0.0  
<

## 5. Conclusions



The dataset is imbalanced, with approximately 8.74% fraudulent
transactions, making accuracy alone an insufficient evaluation metric.

Using the default decision threshold resulted in relatively low recall
for the fraud class. A validation-based threshold analysis was therefore
performed.

A threshold of 0.225 produced the highest F1-score among the evaluated
values and was selected before final test evaluation.

Final test performance:

- Accuracy: 0.9703
- Precision: 0.7939
- Recall: 0.8916
- F1-score: 0.8399
- ROC-AUC: 0.9642
- Average Precision: 0.7289

The selected threshold substantially improved fraud detection while
maintaining relatively high precision.